<a href="https://colab.research.google.com/github/rebhimariem27-tech/Epileptic-Seizure-Prediction-and-Detection-from-Scalp-EEG-/blob/main/EDA_Raw_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
abhishekinnvonix_seizure_epilepcy_chb_mit_eeg_dataset_pediatric_path = kagglehub.dataset_download('abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric')

print('Data source import complete.')


# CHB-MIT (format PhysioNet d'origine) — Exploration des données
**Étape 1/3** : **exploration** → préprocessing / nettoyage → entraînement (prédiction + détection, CSeg-ESP-1 + EpiC-Align).

Dataset Kaggle : `abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric` (dossier `chb-mit-scalp-eeg-database-1.0.0`).
Format attendu : `chbXX/chbXX_YY.edf`, `chbXX/chbXX-summary.txt` (heures des fichiers + début/fin des crises), `*.edf.seizures` (annotations binaires WFDB), `RECORDS`, `RECORDS-WITH-SEIZURES`, `SUBJECT-INFO`.

**Ce que fait ce notebook**
1. Cartographie du dataset (dossiers, types de fichiers, tailles, doublons) : le notebook ne suppose rien sur l'arborescence.
2. Inventaire des EDF (en-têtes lus sans charger le signal) + **intégrité** (taille réelle vs en-tête, fichiers manquants par rapport à `RECORDS`).
3. Lecture des `summary.txt` : heures de début/fin, crises, changements de montage ; contrôle croisé avec les annotations `.seizures`.
4. **Chronologie absolue** par patient (passage de minuit, trous, chevauchements), heures d'enregistrement.
5. Canaux : jeu commun de 18 dérivations bipolaires, canaux dupliqués / vides / référentiels, fichiers où il en manque.
6. Crises : durées, intervalles, crises « leaders », couverture préictale, **déséquilibre** pour les deux protocoles (papier : préictal 5 min / SPH 5 min ; clinique : SOP 30 / SPH 5).
7. Signal : amplitude, canaux plats, ligne 60 Hz, ictal vs interictal par bande.
8. Export des catalogues (`/kaggle/working/exploration/`) réutilisés par le notebook de préprocessing.

**Réglages Kaggle :** Accelerator None, Internet On (pour `pip install mne wfdb`). Durée : quelques minutes.

**Envoie-moi ensuite les sorties** (texte des cellules, ou `exploration/summary.json` + les CSV) : le notebook de préprocessing sera adapté à ce qu'elles montrent.

In [ ]:
import os, re, json, math, time, warnings, collections
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
for pkg in ("mne", "wfdb"):
    try: __import__(pkg)
    except ImportError: os.system(f"pip install -q {pkg}")
import mne; mne.set_log_level("ERROR")
try: import wfdb; HAS_WFDB = True
except Exception: HAS_WFDB = False
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 220); pd.set_option("display.max_rows", 120)

INPUT_BASE = Path(os.environ.get("CHB_INPUT", "/kaggle/input"))
OUT = Path(os.environ.get("CHB_OUT", "/kaggle/working/exploration")); OUT.mkdir(parents=True, exist_ok=True)
MANUAL_ROOT = None        # ex. "/kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0"
PRESETS = {"papier (préictal 5 min, SPH 5)": (5, 5), "clinique (SOP 30 min, SPH 5)": (30, 5)}   # (SOP, SPH) en minutes
INTERICTAL_MIN_DIST_MIN = 60
CANON = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
         "FP2-F8", "F8-T8", "T8-P8", "P8-O2", "FZ-CZ", "CZ-PZ"]
SAME_PATIENT = {"chb21": "chb01"}          # chb21 = même patiente que chb01, 1,5 an plus tard
natkey = lambda s: re.sub(r"\d+", lambda m: m.group().zfill(6), str(s))
print("MNE", mne.__version__, "| wfdb :", HAS_WFDB, "| input :", INPUT_BASE)

## 1. Cartographie du dataset

In [ ]:
rows = []
for dp, dn, fn in os.walk(INPUT_BASE, followlinks=True):
    for f in fn:
        p = Path(dp) / f
        try: sz = p.stat().st_size
        except OSError: sz = np.nan
        rows.append(dict(path=str(p), dir=str(Path(dp).relative_to(INPUT_BASE)), name=f, size=sz,
                         ext=("".join(Path(f).suffixes[-2:]) if f.endswith(".seizures") else Path(f).suffix.lower()) or "(aucune)"))
ALL = pd.DataFrame(rows)
assert len(ALL), "Aucun fichier dans /kaggle/input : Add Input -> le dataset CHB-MIT."
print(f"{len(ALL)} fichiers, {ALL['size'].sum()/1e9:.2f} Go au total dans {INPUT_BASE}\n")
print("Par type de fichier :"); display(ALL.groupby("ext").agg(n=("name", "size"), Go=("size", lambda x: round(x.sum() / 1e9, 2))).sort_values("n", ascending=False))
top = ALL.assign(top=ALL.dir.str.split("/").str[:3].str.join("/")).groupby("top").agg(n=("name", "size"), Go=("size", lambda x: round(x.sum() / 1e9, 2)))
print("Dossiers (3 premiers niveaux) :"); display(top.sort_values("n", ascending=False).head(40))
others = ALL[~ALL.ext.isin([".edf", ".edf.seizures", ".txt"]) & ~ALL.name.isin(["RECORDS", "RECORDS-WITH-SEIZURES", "SUBJECT-INFO"])]
if len(others): print("Autres fichiers (ex. CSV de features fournis par l'auteur du dataset) :"); display(others.head(20)[["dir", "name", "size"]])

## 2. Inventaire et intégrité des EDF (doublons, RECORDS, taille vs en-tête)

In [ ]:
def edf_header(path):
    with open(path, "rb") as f:
        h = f.read(256); ns = int(h[252:256]); n_rec = int(h[236:244]); dur = float(h[244:252])
        sh = f.read(ns * 256)
    labels = [sh[i * 16:(i + 1) * 16].decode("latin1").strip() for i in range(ns)]
    o = ns * (16 + 80 + 8 * 5 + 80)
    nsamp = [int(sh[o + i * 8:o + (i + 1) * 8]) for i in range(ns)]
    sig = [n for n, l in zip(nsamp, labels) if "ANNOTATION" not in l.upper()] or nsamp      # ignorer « EDF Annotations »
    expected = 256 * (ns + 1) + n_rec * sum(nsamp) * 2 if n_rec > 0 else np.nan
    labels = [l for l in labels if "ANNOTATION" not in l.upper()]; nsamp = sig
    return dict(n_signals=len(labels), n_records=n_rec, record_dur=dur, duration_s=n_rec * dur, labels=labels,
                fs=nsamp[0] / dur if dur else np.nan, fs_uniform=len(set(nsamp)) == 1,
                expected_bytes=expected,
                hdr_date=h[168:176].decode("latin1").strip(), hdr_time=h[176:184].decode("latin1").strip())

E = ALL[ALL.ext == ".edf"].copy()
print(f"Fichiers .edf trouvés : {len(E)}" + (f" | exemples : {E.name.head(5).tolist()}" if len(E) else ""))
E["subject"] = E.name.str.lower().str.extract(r"^(chb\d+)", expand=False)
if E.subject.isna().any():
    print(f"   {E.subject.isna().sum()} .edf dont le nom ne commence pas par chbXX (ignorés) : {E[E.subject.isna()].name.head(5).tolist()}")
E = E[E.subject.notna()]
assert len(E), "Aucun EDF chbXX_YY.edf trouvé : regarde la cartographie ci-dessus (le dataset est peut-être en CSV ou zippé)."
dups = E[E.duplicated("name", keep=False)].sort_values("name")
if len(dups):
    same = dups.groupby("name")["size"].nunique().eq(1).all()
    print(f"EDF présents en plusieurs exemplaires : {dups.name.nunique()} noms ({'tailles identiques' if same else 'TAILLES DIFFÉRENTES'}) -> on garde le 1er")
    display(dups.head(10)[["dir", "name", "size"]])
E = E.sort_values("path").drop_duplicates("name").reset_index(drop=True)

def header_mne(path):
    raw = mne.io.read_raw_edf(path, preload=False, verbose="ERROR")
    return dict(n_signals=len(raw.ch_names), n_records=np.nan, record_dur=np.nan, duration_s=raw.n_times / raw.info["sfreq"],
                labels=list(raw.ch_names), fs=raw.info["sfreq"], fs_uniform=True, expected_bytes=np.nan, hdr_date="", hdr_time="")
hdr = []
for p in E.path:
    try: h = edf_header(p)
    except Exception as e1:
        try: h = header_mne(p); h["error"] = f"en-tête brut illisible ({type(e1).__name__}: {e1}) -> lu par MNE"
        except Exception as e2: h = dict(error=f"{type(e1).__name__}: {e1} | MNE: {type(e2).__name__}: {e2}")
    hdr.append(h)
COLS = ["n_signals", "n_records", "record_dur", "duration_s", "labels", "fs", "fs_uniform", "expected_bytes", "hdr_date", "hdr_time", "error"]
H = pd.DataFrame(hdr).reindex(columns=COLS); E = pd.concat([E, H], axis=1)
if E.error.notna().any():
    print(f"\n{E.error.notna().sum()} EDF avec problème de lecture. Exemples :")
    for n_, e_ in E[E.error.notna()][["name", "error"]].head(5).values: print("  ", n_, "->", e_[:300])
    with open(E.path.iloc[0], "rb") as f: print("   256 premiers octets du 1er fichier :", f.read(256))
E["labels"] = E["labels"].map(lambda x: x if isinstance(x, list) else [])
E["bytes_diff"] = E["size"] - E.expected_bytes
E["status"] = np.select([E.duration_s.isna(), E.error.notna(), E.bytes_diff < 0, E.bytes_diff > 0],
                        ["illisible", "lu par MNE", "TRONQUÉ", "octets en trop (OK)"], "ok")
print(f"\nEDF uniques : {len(E)} | sujets : {E.subject.nunique()} -> {sorted(E.subject.unique(), key=natkey)}")
print("Statut d'intégrité :", E.status.value_counts().to_dict())
if (E.status != "ok").any(): display(E[E.status != "ok"][["name", "size", "expected_bytes", "bytes_diff", "status"]])
if (E.status == "illisible").any():
    print(f"-> {int((E.status == 'illisible').sum())} EDF illisibles exclus de la suite de l'analyse")
    E = E[E.status != "illisible"].reset_index(drop=True)
assert len(E), "Aucun EDF lisible : envoie-moi la sortie de cette cellule (exemples d'erreurs et premiers octets)."
print("Fs :", E.fs.value_counts().to_dict(), "| fs non uniforme :", int((~E.fs_uniform.fillna(True)).sum()),
      "| nb de signaux :", E.n_signals.value_counts().to_dict())

def read_list(name):
    m = ALL[ALL.name == name]
    return [l.strip() for l in open(m.path.iloc[0]).read().split("\n") if l.strip()] if len(m) else None
RECORDS, RECSZ = read_list("RECORDS"), read_list("RECORDS-WITH-SEIZURES")
if RECORDS:
    names = {Path(r).name for r in RECORDS}
    print(f"\nRECORDS : {len(names)} fichiers listés | absents du dataset : {sorted(names - set(E.name), key=natkey)[:20]} | en plus : {sorted(set(E.name) - names, key=natkey)[:20]}")
else: print("\nPas de fichier RECORDS")
ROOT = Path(MANUAL_ROOT) if MANUAL_ROOT else Path(os.path.commonpath([str(Path(p).parent.parent) for p in E.path]))
print("Racine CHB-MIT détectée :", ROOT)

## 3. Fichiers `summary.txt` : heures des fichiers, crises, montages — contrôle croisé avec `.seizures`

In [ ]:
def hms(s):
    h, m, sec = [int(x) for x in s.strip().split(":")]; return h * 3600 + m * 60 + sec

SUMS = ALL[ALL.name.str.match(r"chb\d+-summary\.txt$")].sort_values("path").drop_duplicates("name")
file_rows, sz_rows, mont_rows = [], [], []
for p in SUMS.path:
    subj = Path(p).name.split("-")[0]; cur = None; montage = 0; chans = []
    for line in open(p, encoding="latin1"):
        line = line.strip()
        if line.lower().startswith("channels changed") or line.lower().startswith("channels in edf"):
            if chans: mont_rows.append(dict(subject=subj, montage=montage, channels=tuple(chans)))
            montage += 1 if line.lower().startswith("channels changed") else 0; chans = []
        elif re.match(r"Channel \d+:", line): chans.append(line.split(":", 1)[1].strip())
        elif line.startswith("File Name:"):
            cur = dict(subject=subj, name=line.split(":", 1)[1].strip(), start_hms=None, end_hms=None, n_seiz_declared=0, montage=montage)
            file_rows.append(cur)
        elif cur is not None and line.startswith("File Start Time:"): cur["start_hms"] = line.split(":", 1)[1].strip()
        elif cur is not None and line.startswith("File End Time:"): cur["end_hms"] = line.split(":", 1)[1].strip()
        elif cur is not None and line.startswith("Number of Seizures"): cur["n_seiz_declared"] = int(re.findall(r"\d+", line)[-1])
        elif cur is not None and re.match(r"Seizure\s*\d*\s*Start Time", line):
            sz_rows.append(dict(subject=subj, name=cur["name"], onset_s=float(re.findall(r"(\d+)\s*sec", line)[-1])))
        elif cur is not None and re.match(r"Seizure\s*\d*\s*End Time", line):
            sz_rows[-1]["end_s"] = float(re.findall(r"(\d+)\s*sec", line)[-1])
    if chans: mont_rows.append(dict(subject=subj, montage=montage, channels=tuple(chans)))
SF = pd.DataFrame(file_rows); SZ = pd.DataFrame(sz_rows)
SZ["duration_s"] = SZ.end_s - SZ.onset_s
print(f"summary.txt : {len(SUMS)} fichiers | {len(SF)} enregistrements décrits | {len(SZ)} crises")
print("Sujets sans summary :", sorted(set(E.subject) - set(SF.subject), key=natkey))
print("Enregistrements sans heure de début dans le summary :", SF.start_hms.isna().groupby(SF.subject).sum()[lambda x: x > 0].to_dict())
chk = SF.merge(SZ.groupby("name").size().rename("n_parsed"), on="name", how="left").fillna({"n_parsed": 0})
bad = chk[chk.n_seiz_declared != chk.n_parsed]
print("Incohérences nb crises déclaré vs lu :", len(bad)); display(bad.head()) if len(bad) else None
print("EDF sans entrée dans les summary :", sorted(set(E.name) - set(SF.name), key=natkey)[:30])
print("Entrées de summary sans EDF :", sorted(set(SF.name) - set(E.name), key=natkey)[:30])
print("Durées de crise (s) :", SZ.duration_s.describe().round(1).to_dict())
if (SZ.duration_s <= 0).any(): display(SZ[SZ.duration_s <= 0])
MONT = pd.DataFrame(mont_rows)
print("Sujets avec changement de montage :", MONT.groupby("subject").montage.nunique()[lambda x: x > 1].to_dict())

# contrôle croisé avec les annotations binaires .seizures (WFDB)
ANN = ALL[ALL.ext == ".edf.seizures"].sort_values("path").drop_duplicates("name")
if HAS_WFDB and len(ANN):
    xs = []
    import tempfile, shutil as _sh
    tmpd = Path(tempfile.mkdtemp())
    for p in ANN.path:
        rec = p[:-len(".seizures")]
        try:
            _sh.copy(p, tmpd / "rec.seizures")                       # wfdb refuse les noms avec un point (chb01_03.edf)
            a = wfdb.rdann(str(tmpd / "rec"), "seizures"); fs = a.fs or 256
            smp = np.array(a.sample) / fs; sym = list(a.symbol)
            st = [s for s, y in zip(smp, sym) if y == "["]; en = [s for s, y in zip(smp, sym) if y == "]"]
            if not st and len(smp) % 2 == 0: st, en = list(smp[0::2]), list(smp[1::2])
            for o, e in zip(st, en): xs.append(dict(name=Path(rec).name, ann_onset_s=o, ann_end_s=e))
        except Exception as ex: xs.append(dict(name=Path(rec).name, error=str(ex)))
    X = pd.DataFrame(xs)
    print(f"\n.seizures lus : {ANN.shape[0]} fichiers, {X.ann_onset_s.notna().sum() if 'ann_onset_s' in X else 0} crises")
    if "ann_onset_s" in X:
        m = pd.merge_asof(SZ.sort_values("onset_s"), X.dropna(subset=["ann_onset_s"]).sort_values("ann_onset_s"),
                          left_on="onset_s", right_on="ann_onset_s", by="name", direction="nearest", tolerance=30)
        m["d_on"] = (m.onset_s - m.ann_onset_s).abs(); m["d_end"] = (m.end_s - m.ann_end_s).abs()
        print(f"summary vs .seizures : {m.ann_onset_s.notna().sum()}/{len(SZ)} crises retrouvées | écart onset max {m.d_on.max():.1f} s | écart fin max {m.d_end.max():.1f} s")
        print("Fichiers .seizures sans crise dans le summary :", sorted(set(X.name) - set(SZ.name), key=natkey))
    if RECSZ: print("RECORDS-WITH-SEIZURES vs summary :", len({Path(r).name for r in RECSZ}), "listés |", SZ.name.nunique(), "fichiers avec crise dans les summary")
else:
    print("Pas de contrôle croisé .seizures (wfdb absent ou pas de fichiers .seizures)")

## 4. Chronologie absolue par patient
Heures prises dans le summary (`File Start Time`), sinon dans l'en-tête EDF (hh.mm.ss), sinon fichiers supposés consécutifs. Les heures du summary peuvent dépasser 24:00 ; quand l'heure revient en arrière on ajoute un jour.

In [ ]:
R = E.merge(SF[["name", "start_hms", "end_hms", "n_seiz_declared", "montage"]], on="name", how="left")
R["run"] = R.name.str.extract(r"_(\d+)", expand=False).astype(float)
R["hdr_s"] = R.hdr_time.map(lambda t: hms(t.replace(".", ":")) if re.match(r"\d+\.\d+\.\d+", str(t)) else np.nan)
R["sum_s"] = R.start_hms.map(lambda t: hms(t) if isinstance(t, str) and re.match(r"\d+:\d+:\d+", t) else np.nan)
cmp = R.dropna(subset=["hdr_s", "sum_s"]); d = ((cmp.hdr_s - cmp.sum_s) % 86400).clip(upper=86400 - ((cmp.hdr_s - cmp.sum_s) % 86400))
print(f"Heure en-tête EDF = heure summary pour {(d < 2).mean()*100:.0f} % des fichiers comparables")
R["time_source"] = np.where(R.sum_s.notna(), "summary", np.where(R.hdr_s.notna(), "edf_header", "consecutive"))
R = R.sort_values(["subject", "name"], key=lambda c: c.map(natkey)).reset_index(drop=True)
R["t0_abs_s"] = np.nan
for s, g in R.groupby("subject"):
    day, prev_end, out = 0, None, []
    for r in g.itertuples():
        tod = r.sum_s if not np.isnan(r.sum_s) else r.hdr_s
        if np.isnan(tod):
            t0 = 0.0 if prev_end is None else prev_end
        else:
            t0 = tod + day * 86400
            while prev_end is not None and t0 < prev_end - 3 * 3600: day += 1; t0 += 86400
        out.append(t0); prev_end = t0 + r.duration_s
    R.loc[g.index, "t0_abs_s"] = out
R["t1_abs_s"] = R.t0_abs_s + R.duration_s
R["gap_prev_s"] = R.t0_abs_s - R.groupby("subject").t1_abs_s.shift(1)
# heures d'en-tête incohérentes (fichiers qui se chevauchent) -> fichiers supposés consécutifs pour ce sujet
for s in R[(R.gap_prev_s < -60) & (R.time_source != "summary")].subject.unique():
    g = R[R.subject == s]; t0 = np.concatenate([[0.0], np.cumsum(g.duration_s.values)[:-1]])
    R.loc[g.index, "t0_abs_s"] = t0; R.loc[g.index, "time_source"] = "consecutive"
    print(f"{s} : heures d'en-tête incohérentes -> fichiers supposés consécutifs (trous inconnus)")
R["t1_abs_s"] = R.t0_abs_s + R.duration_s
R["gap_prev_s"] = R.t0_abs_s - R.groupby("subject").t1_abs_s.shift(1)
print("Source de l'heure :", R.time_source.value_counts().to_dict())
print("Écarts entre fichiers consécutifs (s) :", R.gap_prev_s.describe().round(0).to_dict())
print("Chevauchements (> 1 s) :", int((R.gap_prev_s < -1).sum()), "| trous > 1 h :", int((R.gap_prev_s > 3600).sum()))
if (R.gap_prev_s < -1).any(): display(R[R.gap_prev_s < -1][["subject", "name", "start_hms", "gap_prev_s"]].head(10))

SZ = SZ.merge(R[["name", "t0_abs_s", "t1_abs_s"]], on="name", how="left")
SZ["onset_abs_s"] = SZ.t0_abs_s + SZ.onset_s; SZ["end_abs_s"] = SZ.t0_abs_s + SZ.end_s
SZ = SZ.sort_values(["subject", "onset_abs_s"], key=lambda c: c.map(natkey) if c.name == "subject" else c).reset_index(drop=True)
SZ["seizure_id"] = SZ.subject + "_sz" + SZ.groupby("subject").cumcount().astype(str).str.zfill(2)
SZ["prev_gap_s"] = (SZ.onset_abs_s - SZ.groupby("subject").end_abs_s.shift(1)).fillna(np.inf)
SZ["group"] = SZ.subject.map(lambda s: SAME_PATIENT.get(s, s))

## 5. Résumé par sujet

In [ ]:
info = ALL[ALL.name == "SUBJECT-INFO"]
SI = pd.DataFrame([dict(subject=m[0], gender=m[1], age=float(m[2])) for m in
                   re.findall(r"(chb\d+)\s+([MF])\s+([\d.]+)", open(info.path.iloc[0]).read())]) if len(info) else pd.DataFrame()
SUB = R.groupby("subject").agg(n_files=("name", "size"), hours=("duration_s", lambda x: x.sum() / 3600),
                               n_signals=("n_signals", lambda x: "/".join(map(str, sorted(set(x))))),
                               time_source=("time_source", lambda x: "/".join(sorted(set(x))))).reset_index()
SUB = SUB.merge(SZ.groupby("subject").agg(n_seizures=("seizure_id", "size"), seiz_dur_mean_s=("duration_s", "mean")).reset_index(), on="subject", how="left").fillna({"n_seizures": 0})
if len(SI): SUB = SUB.merge(SI, on="subject", how="left")
SUB["group"] = SUB.subject.map(lambda s: SAME_PATIENT.get(s, s))
SUB = SUB.sort_values("subject", key=lambda c: c.map(natkey)).reset_index(drop=True)
display(SUB.round(2))
print(f"TOTAL : {R.duration_s.sum()/3600:.1f} h | {len(SZ)} crises | ictal = {SZ.duration_s.sum()/60:.1f} min "
      f"({100*SZ.duration_s.sum()/R.duration_s.sum():.3f} % du temps) | {SUB.group.nunique()} patients distincts")
fig, ax = plt.subplots(1, 4, figsize=(22, 4.5))
SUB.set_index("subject").hours.plot.bar(ax=ax[0], color="steelblue"); ax[0].set_title("Heures / sujet")
SUB.set_index("subject").n_seizures.plot.bar(ax=ax[1], color="crimson"); ax[1].set_title("Crises / sujet")
ax[2].hist(R.duration_s / 60, bins=40); ax[2].set_title("Durée des fichiers (min)")
ax[3].hist(SZ.duration_s, bins=40, color="crimson"); ax[3].set_title("Durée des crises (s)")
plt.tight_layout(); plt.savefig(OUT / "overview.png", dpi=100); plt.show()

## 6. Canaux : jeu commun de 18 dérivations, doublons, canaux vides, montages référentiels

In [ ]:
def norm(l): return re.sub(r"^EEG", "", l.upper().replace(" ", ""))
def canon_status(labels):
    lab = [norm(l) for l in labels]; base = [re.sub(r"-\d+$", "", l) if re.fullmatch(r"[A-Z0-9]+-[A-Z0-9]+-\d+", l) else l for l in lab]
    refs = {re.sub(r"-(REF|CS2|AVG|LE|AR)$", "", l) for l in lab if re.fullmatch(r"[A-Z0-9]+(-(REF|CS2|AVG|LE|AR))?", l)}
    direct = [c for c in CANON if c in base]
    derivable = [c for c in CANON if c not in base and all(e in refs for e in c.split("-"))]
    return len(direct), len(derivable), [c for c in CANON if c not in base and c not in derivable], collections.Counter(base)
st = R.labels.map(canon_status)
R["n_canon_direct"] = st.map(lambda x: x[0]); R["n_canon_derivable"] = st.map(lambda x: x[1]); R["canon_missing"] = st.map(lambda x: x[2])
R["has_dup_label"] = st.map(lambda x: any(v > 1 for k, v in x[3].items() if k not in ("-", ".", "")))
R["n_empty_ch"] = R.labels.map(lambda L: sum(norm(l) in ("-", ".", "") for l in L))
CNT = collections.Counter(norm(l) for L in R.labels for l in set(L) if "ANNOTATION" not in l.upper())
CH = pd.DataFrame({"channel": list(CNT), "n_files": list(CNT.values())}).assign(pct=lambda d: (100 * d.n_files / len(R)).round(1)).sort_values("n_files", ascending=False)
display(CH.head(45).T)
print("Fichiers avec les 18 dérivations directement :", int((R.n_canon_direct == 18).sum()), "/", len(R))
print("Fichiers complétables par dérivation référentielle :", int(((R.n_canon_direct < 18) & (R.n_canon_direct + R.n_canon_derivable == 18)).sum()))
inc = R[R.n_canon_direct + R.n_canon_derivable < 18]
print("Fichiers où il manque des dérivations (non récupérables) :", len(inc))
if len(inc): display(inc[["subject", "name", "n_signals", "canon_missing"]].head(25))
print("Fichiers avec label dupliqué (ex. T8-P8) :", int(R.has_dup_label.sum()), "| avec canaux vides '-' ou '.' :", int((R.n_empty_ch > 0).sum()))
print("Exemple de lecture MNE (noms après déduplication) :", mne.io.read_raw_edf(R.path.iloc[0], preload=False).ch_names)

## 7. Crises : intervalles, crises « leaders », couverture préictale, déséquilibre de classes

In [ ]:
def covered(sub, a, b):
    g = R[R.subject == sub]; return float(np.clip(np.minimum(g.t1_abs_s, b) - np.maximum(g.t0_abs_s, a), 0, None).sum())
TOT = R.duration_s.sum(); imb = []
for name, (sop, sph) in PRESETS.items():
    SOP, SPH = sop * 60, sph * 60
    lead = SZ.prev_gap_s >= SOP + SPH
    cov = np.array([covered(r.subject, r.onset_abs_s - SPH - SOP, r.onset_abs_s - SPH) for r in SZ.itertuples()]) / SOP
    SZ[f"lead_{sop}_{sph}"] = lead; SZ[f"cov_{sop}_{sph}"] = cov
    ok = lead & (cov >= 0.5)
    pre_s = (cov * SOP)[ok].sum(); ict = SZ.duration_s.sum()
    near = sum(covered(s, o - INTERICTAL_MIN_DIST_MIN * 60, e + INTERICTAL_MIN_DIST_MIN * 60) for s, o, e in zip(SZ.subject, SZ.onset_abs_s, SZ.end_abs_s))
    inter = TOT - min(near, TOT)
    print(f"{name} : crises leaders {int(lead.sum())}/{len(SZ)} | utilisables (couverture >= 50 %) {int(ok.sum())} | "
          f"préictal {pre_s/3600:.1f} h | ictal {ict/3600:.2f} h | interictal (>= {INTERICTAL_MIN_DIST_MIN} min) ≈ {inter/3600:.0f} h "
          f"-> ratio interictal:préictal ≈ {inter/max(pre_s,1):.0f}:1")
    imb.append(dict(preset=name, lead=int(lead.sum()), usable=int(ok.sum()), preictal_h=pre_s / 3600, ictal_h=ict / 3600, interictal_h=inter / 3600))
per = SZ.groupby("group").agg(crises=("seizure_id", "size"), **{f"utilisables_{k}": (f"lead_{s}_{p}", "sum") for k, (s, p) in zip(["papier", "clinique"], PRESETS.values())})
display(per.T)
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
SZ.prev_gap_s.replace(np.inf, np.nan).dropna().div(60).clip(upper=1500).plot.hist(bins=50, ax=ax[0]); ax[0].set_title("Intervalle depuis la crise précédente (min, borné à 1500)")
for i, (s, g) in enumerate(R.groupby("subject", sort=False)):
    ax[1].barh(i, g.duration_s / 3600, left=g.t0_abs_s / 3600, color="lightsteelblue")
    ax[1].scatter(SZ[SZ.subject == s].onset_abs_s / 3600, [i] * (SZ.subject == s).sum(), c="red", s=8, zorder=3)
ax[1].set_yticks(range(R.subject.nunique())); ax[1].set_yticklabels(R.subject.unique(), fontsize=6); ax[1].set_title("Chronologie (rouge = crise)"); ax[1].set_xlabel("h")
plt.tight_layout(); plt.savefig(OUT / "seizures_timeline.png", dpi=100); plt.show()

## 8. Signal : amplitude, canaux plats, ligne 60 Hz, ictal vs interictal (1 fichier avec crise par sujet, max 6)

In [ ]:
from scipy.signal import welch
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 55)}
sig_rows = []
for s, g in list(SZ.groupby("subject", sort=False))[:6]:
    z = g.iloc[0]; path = R.loc[R.name == z["name"], "path"].iloc[0]
    raw = mne.io.read_raw_edf(path, preload=True); X = raw.get_data() * 1e6; fs = raw.info["sfreq"]; names = raw.ch_names
    keep = [i for i, n in enumerate(names) if re.sub(r"-\d+$", "", n.upper()) in CANON]; X, names = X[keep], [names[i] for i in keep]
    a, b = int(z.onset_s * fs), int(z.end_s * fs)
    f, P = welch(X, fs=fs, nperseg=int(4 * fs), axis=-1)
    line = P[:, (f > 59) & (f < 61)].mean() / P[:, (f > 50) & (f < 58)].mean()
    flat = [names[i] for i in np.where(X.std(1) < 0.5)[0]]
    m2 = int(120 * fs); before, after = (0, a - m2), (b + m2, X.shape[1])          # segment de référence loin de la crise, dans le même fichier
    lo_, hi_ = max(before, after, key=lambda x: x[1] - x[0]); hi_ = min(hi_, lo_ + int(300 * fs))
    if hi_ - lo_ >= int(30 * fs) and b - a >= int(4 * fs):
        fi, Pi = welch(X[:, a:b], fs=fs, nperseg=int(2 * fs), axis=-1); fn, Pn = welch(X[:, lo_:hi_], fs=fs, nperseg=int(2 * fs), axis=-1)
        r = {k: float(np.median(Pi[:, (fi >= lo) & (fi < hi)].mean(-1) / (Pn[:, (fn >= lo) & (fn < hi)].mean(-1) + 1e-12))) for k, (lo, hi) in BANDS.items()}
    else: r = {k: np.nan for k in BANDS}
    sig_rows.append(dict(subject=s, file=z["name"], amp_median_uV=float(np.median(np.abs(X))), p99_uV=float(np.percentile(np.abs(X), 99)),
                         flat_channels=flat, line60_ratio=float(line), **{f"ictal/inter_{k}": v for k, v in r.items()}))
    s0, s1 = max(0, a - int(60 * fs)), min(X.shape[1], b + int(60 * fs)); t = np.arange(s0, s1) / fs
    fig, ax = plt.subplots(figsize=(18, 5))
    for i in range(min(10, len(names))): ax.plot(t, X[i, s0:s1] - i * 300, lw=.4)
    ax.axvspan(z.onset_s, z.end_s, color="red", alpha=.12); ax.set_yticks([-i * 300 for i in range(min(10, len(names)))]); ax.set_yticklabels(names[:10])
    ax.set_title(f"{z['name']} — crise {z.onset_s:.0f}–{z.end_s:.0f} s"); plt.tight_layout(); plt.show()
SIGQ = pd.DataFrame(sig_rows); display(SIGQ.round(2))
print("line60_ratio >> 1 = forte ligne 60 Hz -> notch utile ; ictal/inter > 1 = puissance plus forte pendant la crise")

## 9. Export des catalogues pour le notebook de préprocessing

In [ ]:
R.assign(labels=R.labels.map("|".join), canon_missing=R.canon_missing.map("|".join)).to_csv(OUT / "recordings.csv", index=False)
SZ.to_csv(OUT / "seizures.csv", index=False); SUB.to_csv(OUT / "subjects.csv", index=False); CH.to_csv(OUT / "channels.csv", index=False)
summary = dict(root=str(ROOT), n_files_total=int(len(ALL)), total_gb=float(ALL["size"].sum() / 1e9), n_edf=int(len(E)),
               n_edf_duplicates_removed=int(dups.name.nunique()) if len(dups) else 0, edf_status=E.status.value_counts().to_dict(),
               records_listed=len(RECORDS) if RECORDS else None, n_subjects=int(E.subject.nunique()), n_patients=int(SUB.group.nunique()),
               hours=float(R.duration_s.sum() / 3600), n_seizures=int(len(SZ)), time_source=R.time_source.value_counts().to_dict(),
               overlaps=int((R.gap_prev_s < -1).sum()), fs=sorted(map(float, E.fs.dropna().unique())),
               files_all18_direct=int((R.n_canon_direct == 18).sum()), files_missing_canon=int(len(inc)),
               montage_changes=MONT.groupby("subject").montage.nunique()[lambda x: x > 1].to_dict(), imbalance=imb,
               other_files=others.name.head(20).tolist())
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=str)
print(json.dumps(summary, indent=1, default=str)); print("\nFichiers exportés :", sorted(os.listdir(OUT)))

## 8 bis. Visualisations du signal **brut** (EDA)

Uniquement les données telles que fournies : **aucun filtre, aucun rejet de fenêtre, aucune règle de sélection** (pas de SOP/SPH, pas de « leaders »).

* **8 bis.1 — niveau dataset** : instantané à partir des tableaux `R` (enregistrements) et `SZ` (crises), aucun signal chargé.
* **8 bis.2 — niveau signal** : une crise par patient (`VZ["n_subjects"]` patients) est chargée une fois, en brut (≈ 1 à 3 min). Segments : *5 min avant la crise*, *crise*, *référence hors crise* (même fichier, ≥ 2 min de la crise).

Réglages dans le dictionnaire `VZ`. Toutes les figures sont sauvegardées dans `OUT/figures/`.

In [ ]:
# ======================= 8 bis. EDA SIGNAL BRUT — réglages et outils =======================
from scipy.signal import welch, spectrogram
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)

VZ = dict(win_s=5.0, n_subjects=8, seg_s=300)   # fenêtre d'analyse (s), nb de patients échantillonnés, durée des segments (s)
VZ_BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 58)}   # gamma s'arrête avant la ligne secteur 60 Hz
C_RAW, C_PRE, C_ICT, C_INT = "#d95f02", "#1f77b4", "#d62728", "#7f7f7f"


def vz_save(name): plt.savefig(FIG / f"{name}.png", dpi=110, bbox_inches="tight")


def vz_pick18(raw):
    """Indices MNE des 18 dérivations CANON (dans l'ordre CANON) ; None si l'une manque. Sélection de canaux uniquement."""
    first = {}
    for i, n in enumerate(raw.ch_names):
        k = re.sub(r"^EEG", "", n.upper().replace(" ", ""))
        m = re.fullmatch(r"([A-Z0-9]+-[A-Z0-9]+)-\d+", k); first.setdefault(m.group(1) if m else k, i)
    return [first[c] for c in CANON] if all(c in first for c in CANON) else None


def vz_bandpow(X, fs):
    f, P = welch(X, fs=fs, nperseg=int(2 * fs), axis=-1)
    return np.stack([np.log10(P[:, (f >= lo) & (f < hi)].mean(-1) + 1e-12) for lo, hi in VZ_BANDS.values()], 1)   # (18, 5), log10 µV²/Hz


def vz_windows(X, fs):
    n = int(VZ["win_s"] * fs); k = X.shape[1] // n
    W = X[:, :k * n].reshape(X.shape[0], k, n).transpose(1, 0, 2)
    return dict(maxabs=np.abs(W).max((1, 2)), p2p=np.ptp(W, 2).max(1), lstd=np.log10(W.std((1, 2)) + 1e-9))


def vz_line_db(X, fs):
    """Puissance à 59–61 Hz relative à 1–30 Hz (dB) : mesure de la ligne secteur."""
    f, P = welch(X, fs=fs, nperseg=int(4 * fs), axis=-1)
    return float(10 * np.log10(P[:, (f > 59) & (f < 61)].mean() / P[:, (f >= 1) & (f <= 30)].mean() + 1e-15))


# positions des 18 dérivations bipolaires = milieu des deux électrodes (montage standard 10-20)
_mont = mne.channels.make_standard_montage("standard_1020")
_pos = {k.upper(): np.array(v) for k, v in _mont.get_positions()["ch_pos"].items()}
VZ_XY = np.array([((_pos[c.split("-")[0]] + _pos[c.split("-")[1]]) / 2)[:2] for c in CANON])


def vz_head(ax, vals, vlim, cmap):
    im, _ = mne.viz.plot_topomap(vals, VZ_XY, axes=ax, show=False, contours=0, sphere=0.11, extrapolate="head", cmap=cmap, vlim=vlim)
    return im


print(f"EDA brut | figures -> {FIG}")

### 8 bis.1 — Répartition des données brutes (aucun signal chargé)

In [ ]:
# ======================= 8 bis.1  NIVEAU DATASET =======================
rows = []
for g, zs in SZ.groupby("group"):
    subs = sorted(set(zs.subject)); r = R[R.subject.isin(subs)]
    rows.append(dict(group=g, n_files=len(r), recorded_h=r.duration_s.sum() / 3600, seizures=len(zs),
                     ictal_min=zs.duration_s.sum() / 60, files_18ch=int((r.n_canon_direct == 18).sum())))
VZ_TAB = pd.DataFrame(rows).sort_values("group", key=lambda c: c.map(natkey)).reset_index(drop=True)
VZ_TAB["ictal_pct"] = 100 * VZ_TAB.ictal_min / (VZ_TAB.recorded_h * 60)
display(VZ_TAB.round(2).T)

x = np.arange(len(VZ_TAB)); lbl = VZ_TAB.group.str.replace("chb", "")
fig, ax = plt.subplots(2, 3, figsize=(21, 10))
ax[0, 0].bar(x, VZ_TAB.seizures, color=C_ICT); ax[0, 0].set_title("Nombre de crises par patient")
ax[0, 1].bar(x, VZ_TAB.recorded_h, color=C_RAW); ax[0, 1].set_title("Heures enregistrées par patient")
ax[0, 2].bar(x, VZ_TAB.ictal_pct, color=C_ICT); ax[0, 2].set_title("Part du temps en crise (%) — déséquilibre des classes")
for a in ax[0]: a.set_xticks(x); a.set_xticklabels(lbl); a.set_xlabel("patient")
bins = np.logspace(np.log10(max(SZ.duration_s.min(), 1)), np.log10(SZ.duration_s.max()), 30)
ax[1, 0].hist(SZ.duration_s, bins=bins, color=C_ICT, alpha=.8); ax[1, 0].set_xscale("log"); ax[1, 0].set_xlabel("s"); ax[1, 0].set_title(f"Durée des crises ({len(SZ)} crises)")
gap = SZ.prev_gap_s.replace(np.inf, np.nan).dropna() / 60
ax[1, 1].hist(gap, bins=np.logspace(np.log10(max(gap.min(), .5)), np.log10(gap.max()), 40), color=C_RAW, alpha=.8)
ax[1, 1].set_xscale("log"); ax[1, 1].set_xlabel("min"); ax[1, 1].set_title("Intervalle depuis la crise précédente (même patient)")
ax[1, 2].bar(x, VZ_TAB.n_files, color="#bbbbbb", label="fichiers EDF"); ax[1, 2].bar(x, VZ_TAB.files_18ch, color=C_PRE, label="dont 18 dérivations directes")
ax[1, 2].set_xticks(x); ax[1, 2].set_xticklabels(lbl); ax[1, 2].set_title("Fichiers EDF par patient"); ax[1, 2].legend(fontsize=8)
plt.tight_layout(); vz_save("dataset_brut"); plt.show()

# --- heure du jour : temps enregistré vs début des crises ---
t0k = R[R.time_source != "consecutive"]
hrs = np.zeros(24)
for a, d in zip(t0k.t0_abs_s, t0k.duration_s): hrs += np.bincount((((a + np.arange(0, d, 60)) % 86400) // 3600).astype(int), minlength=24)
hrs = hrs / hrs.sum(); sz_t = SZ.merge(R[["name", "time_source"]], on="name"); sz_t = sz_t[sz_t.time_source != "consecutive"]
h_on = np.bincount(((sz_t.onset_abs_s % 86400) // 3600).astype(int), minlength=24); h_on = h_on / h_on.sum()
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8)); hh = np.arange(24)
ax[0].bar(hh - .2, hrs, .4, color=C_RAW, label="temps enregistré"); ax[0].bar(hh + .2, h_on, .4, color=C_ICT, label="début des crises")
ax[0].set_title("Heure du jour (risque de raccourci jour/nuit)"); ax[0].set_xlabel("heure"); ax[0].legend(fontsize=8)
for r_ in VZ_TAB.itertuples(): ax[1].scatter(r_.recorded_h, r_.seizures, s=40, color=C_PRE); ax[1].annotate(r_.group.replace("chb", ""), (r_.recorded_h, r_.seizures), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax[1].set_xlabel("heures enregistrées"); ax[1].set_ylabel("crises"); ax[1].set_title("Crises vs durée d'enregistrement (1 point = 1 patient)")
plt.tight_layout(); vz_save("dataset_heure_brut"); plt.show()
VZ_TAB.to_csv(OUT / "viz_dataset_brut.csv", index=False)
print(f"{VZ_TAB.recorded_h.sum():.0f} h | {len(SZ)} crises | {VZ_TAB.ictal_min.sum():.0f} min en crise ({100*VZ_TAB.ictal_min.sum()/(VZ_TAB.recorded_h.sum()*60):.2f} % du temps)")

### 8 bis.2 — Signal brut : échantillon de crises chargé une fois
Une crise par patient (fichiers avec les 18 dérivations directes). Segments : **5 min avant la crise**, **crise** (60 s max), **référence hors crise** (même fichier, ≥ 2 min de la crise). Signal en µV, tel quel dans l'EDF.

In [ ]:
# ======================= 8 bis.2  ÉCHANTILLON SIGNAL BRUT =======================
def vz_choose():
    z = SZ[(SZ.onset_s >= VZ["seg_s"] + 120) & (SZ.duration_s >= 8)]
    z = z.merge(R[["name", "path", "n_canon_direct"]], on="name"); z = z[z.n_canon_direct == 18]
    z = z.assign(_k=z.subject.map(natkey)).sort_values(["_k", "onset_abs_s"]).drop_duplicates("group")
    return z.iloc[np.unique(np.linspace(0, len(z) - 1, min(VZ["n_subjects"], len(z))).round().astype(int))]


def vz_build(row):
    raw = mne.io.read_raw_edf(row.path, preload=True, verbose="ERROR"); idx = vz_pick18(raw)
    if idx is None: return None
    fs = raw.info["sfreq"]; X = (raw.get_data(picks=idx) * 1e6).astype(np.float32); T = X.shape[1] / fs
    on, en = row.onset_s, row.end_s; L = VZ["seg_s"]
    seg = {"pre": (on - L, on), "ict": (on, min(en, on + 60))}
    free = [(10.0, seg["pre"][0] - 120), (en + 120, T)]; a, b = max(free, key=lambda t: t[1] - t[0])
    if b - a < 60: return None
    seg["int"] = (a, min(b, a + L)); sl = (max(0.0, on - 1800), min(T, on + 600))
    cut = lambda ab: X[:, int(ab[0] * fs):int(ab[1] * fs)].copy()
    S = dict(subject=row.subject, group=row.group, name=row.name, fs=fs, on=on, en=en, seg=seg, sl=sl,
             sig={k: cut(v) for k, v in seg.items()}, slice=cut(sl))
    S["bp"] = {k: vz_bandpow(v, fs) for k, v in S["sig"].items()}
    S["win"] = vz_windows(S["slice"], fs)
    S["line"] = vz_line_db(S["sig"]["int"], fs)
    S["psd"] = welch(S["sig"]["int"], fs=fs, nperseg=int(4 * fs), axis=-1)
    del raw, X; return S


VZ_S = []; t_ = time.time()
for row in vz_choose().itertuples():
    try: s_ = vz_build(row)
    except Exception as ex_: print(f"  ! {row.name} ignoré : {type(ex_).__name__}: {ex_}"); continue
    if s_ is not None: VZ_S.append(s_); print(f"  ✔ {row.name} ({row.subject}) crise {row.onset_s:.0f}–{row.end_s:.0f} s")
print(f"{len(VZ_S)} patients dans l'échantillon signal ({time.time() - t_:.0f} s)")
assert VZ_S, "Aucune crise exploitable pour l'échantillon signal (voir les messages ci-dessus)."

#### Spectrogramme brut autour d'une crise
Haut : spectrogramme (moyenne des 18 dérivations). Bas : puissance par bande dans le temps, relative au niveau hors crise. Bande bleue = 5 min avant la crise ; rouge = crise.

In [ ]:
def vz_spec(X, fs, fmax=70):
    nper, hop, acc = int(4 * fs), int(2 * fs), None
    for ch in X:
        f, t, S = spectrogram(ch, fs=fs, nperseg=nper, noverlap=nper - hop, detrend="constant", mode="psd")
        S = S[f <= fmax]; acc = S if acc is None else acc + S
    return f[f <= fmax], t, acc / len(X)


def plot_spectrograms(i=0):
    s = VZ_S[i]; fs = s["fs"]; f, t, S = vz_spec(s["slice"], fs); tm = (t + s["sl"][0] - s["on"]) / 60; db = 10 * np.log10(S + 1e-12)
    fig, ax = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw=dict(height_ratios=[1, .8]))
    im = ax[0].pcolormesh(tm, f, db, shading="auto", cmap="magma", vmin=np.percentile(db, 2), vmax=np.percentile(db, 99.5))
    ax[0].set_ylabel("Hz"); ax[0].set_title("EDF brut — moyenne des 18 dérivations", loc="left"); fig.colorbar(im, ax=ax[0], label="dB (µV²/Hz)", pad=.01)
    ax[0].axhline(60, color="cyan", lw=.8, ls=":"); ax[0].text(tm[0], 61, "ligne secteur 60 Hz", color="cyan", fontsize=8)
    ref = tm < (s["seg"]["pre"][0] - s["on"]) / 60 - 2
    for (b, (lo, hi)), col in zip(VZ_BANDS.items(), plt.cm.viridis(np.linspace(0, .9, 5))):
        pb = 10 * np.log10(S[(f >= lo) & (f < hi)].sum(0) + 1e-12); ax[1].plot(tm, pb - np.median(pb[ref] if ref.any() else pb), color=col, label=b)
    for a in ax:
        a.axvspan((s["seg"]["pre"][0] - s["on"]) / 60, 0, color=C_PRE, alpha=.18); a.axvspan(0, (s["en"] - s["on"]) / 60, color=C_ICT, alpha=.25)
    ax[1].set_ylabel("dB vs hors crise"); ax[1].set_xlabel("minutes depuis le début de crise"); ax[1].set_title("Puissance par bande dans le temps", loc="left"); ax[1].legend(ncol=5, fontsize=8)
    fig.suptitle(f"{s['name']} — crise à {s['on']:.0f} s", y=.995); plt.tight_layout(); vz_save(f"spectrogramme_{s['subject']}"); plt.show()


plot_spectrograms(0)          # plot_spectrograms(k) pour un autre patient (k = 0 … len(VZ_S)-1)

#### Activité de la tête (cartes topographiques par bande)
Chaque dérivation bipolaire est placée au **milieu de ses deux électrodes** (10-20 standard) : cartes approximatives, pour comparer, pas pour localiser une source.

In [ ]:
def plot_topomaps(i=0):
    s = VZ_S[i]; bp = s["bp"]
    rows = [("hors crise (log10 µV²/Hz)", bp["int"], "viridis"), ("avant crise (log10 µV²/Hz)", bp["pre"], "viridis"), ("crise (log10 µV²/Hz)", bp["ict"], "viridis"),
            ("avant crise / hors crise (log10)", bp["pre"] - bp["int"], "RdBu_r"), ("crise / hors crise (log10)", bp["ict"] - bp["int"], "RdBu_r")]
    fig, ax = plt.subplots(5, 5, figsize=(17, 17), layout="constrained")
    for j, b in enumerate(VZ_BANDS):
        allv = np.r_[bp["int"][:, j], bp["pre"][:, j], bp["ict"][:, j]]; vl = (allv.min(), allv.max())
        im = [vz_head(ax[k, j], rows[k][1][:, j], vl, "viridis") for k in range(3)]; fig.colorbar(im[0], ax=ax[:3, j], location="bottom", shrink=.8, pad=.02)
        for k in (3, 4):
            m = max(np.abs(rows[k][1][:, j]).max(), 1e-3); im = vz_head(ax[k, j], rows[k][1][:, j], (-m, m), "RdBu_r"); fig.colorbar(im, ax=ax[k, j], location="bottom", shrink=.8, pad=.02)
        ax[0, j].set_title(f"{b} ({VZ_BANDS[b][0]}–{VZ_BANDS[b][1]} Hz)", fontsize=12)
    for k in range(5): ax[k, 0].set_ylabel(rows[k][0], fontsize=9)
    fig.suptitle(f"{s['name']} — puissance par bande, signal brut", fontsize=13); vz_save(f"topomap_{s['subject']}"); plt.show()


def plot_topomaps_group():
    mean = lambda a, b: np.mean([s["bp"][a] - s["bp"][b] for s in VZ_S], 0)
    rows = [("avant crise / hors crise", mean("pre", "int")), ("crise / hors crise", mean("ict", "int"))]
    fig, ax = plt.subplots(2, 5, figsize=(17, 7.5), layout="constrained")
    for j, b in enumerate(VZ_BANDS):
        m = max(np.abs(r[1][:, j]).max() for r in rows) or 1e-3
        for k in range(2): im = vz_head(ax[k, j], rows[k][1][:, j], (-m, m), "RdBu_r")
        fig.colorbar(im, ax=ax[:, j], location="bottom", shrink=.8, pad=.02, label="log10 ratio"); ax[0, j].set_title(b, fontsize=12)
    for k in range(2): ax[k, 0].set_ylabel(rows[k][0], fontsize=9)
    fig.suptitle(f"Moyenne sur {len(VZ_S)} patients (signal brut)", fontsize=13); vz_save("topomap_groupe"); plt.show()


plot_topomaps(0); plot_topomaps_group()

#### Distributions du signal brut
(a) amplitude des fenêtres de 5 s par patient · (b) amplitude max des fenêtres · (c) spectre moyen par patient (hors crise) · (d) ligne secteur · (e) puissance par classe.
Une forte hétérogénéité d'amplitude **entre patients** est à anticiper pour la suite (normalisation).

In [ ]:
def plot_distributions():
    subs = [s["subject"] for s in VZ_S]; xi = np.arange(len(VZ_S)); fig, ax = plt.subplots(2, 3, figsize=(21, 10))
    bp = ax[0, 0].boxplot([s["win"]["lstd"] for s in VZ_S], positions=xi, widths=.5, patch_artist=True, showfliers=False)
    for b in bp["boxes"]: b.set_facecolor(C_RAW); b.set_alpha(.7)
    ax[0, 0].set_xticks(xi); ax[0, 0].set_xticklabels(subs, rotation=45); ax[0, 0].set_ylabel("log10 écart-type de la fenêtre (µV)"); ax[0, 0].set_title("(a) Amplitude par patient")
    allr = np.concatenate([s["win"]["maxabs"] for s in VZ_S])
    ax[0, 1].hist(allr, bins=np.logspace(0, np.log10(allr.max() * 1.2), 60), color=C_RAW, alpha=.8)
    ax[0, 1].set_xscale("log"); ax[0, 1].set_xlabel("amplitude max de la fenêtre (µV)"); ax[0, 1].set_title(f"(b) Fenêtres de {VZ['win_s']:.0f} s : {len(allr)} dans l'échantillon")
    for s, col in zip(VZ_S, plt.cm.tab10(np.arange(len(VZ_S)) % 10)):
        f, P = s["psd"]; ax[0, 2].semilogy(f, P.mean(0), color=col, lw=1, label=s["subject"])
    ax[0, 2].set_xlim(0, 100); ax[0, 2].set_xlabel("Hz"); ax[0, 2].set_title("(c) Spectre moyen hors crise"); ax[0, 2].legend(fontsize=7, ncol=2)
    ax[1, 0].bar(xi, [s["line"] for s in VZ_S], .6, color=C_RAW); ax[1, 0].set_xticks(xi); ax[1, 0].set_xticklabels(subs, rotation=45)
    ax[1, 0].set_ylabel("dB (59–61 Hz relatif à 1–30 Hz)"); ax[1, 0].set_title("(d) Ligne secteur 60 Hz")
    cls = [("int", "hors crise", C_INT), ("pre", "avant crise", C_PRE), ("ict", "crise", C_ICT)]; rng = np.random.default_rng(0)
    for k, (c, nm, col) in enumerate(cls):
        v = [np.median(s["bp"][c]) for s in VZ_S]; ax[1, 1].scatter(k + rng.uniform(-.08, .08, len(v)), v, color=col, alpha=.8)
    ax[1, 1].set_xticks(range(3)); ax[1, 1].set_xticklabels([c[1] for c in cls]); ax[1, 1].set_ylabel("log10 puissance médiane (µV²/Hz)"); ax[1, 1].set_title("(e) Puissance par classe — 1 point = 1 patient")
    ax[1, 2].axis("off"); plt.tight_layout(); vz_save("distributions_signal_brut"); plt.show()
    display(pd.DataFrame(dict(patient=subs, amp_mediane_uV=[10 ** np.median(s["win"]["lstd"]) for s in VZ_S],
                              amp_max_uV=[s["win"]["maxabs"].max() for s in VZ_S], ligne_secteur_dB=[s["line"] for s in VZ_S])).round(2))


plot_distributions()

#### Canaux : variabilité par dérivation (signal brut)

In [ ]:
def plot_channel_heatmaps():
    M = np.array([np.log10(s["slice"].std(1) + 1e-9) for s in VZ_S])
    fig, ax = plt.subplots(1, 2, figsize=(18, 5.5), gridspec_kw=dict(width_ratios=[1.3, .8]))
    im = ax[0].imshow(M, aspect="auto", cmap="viridis"); ax[0].set_title("log10 écart-type par canal (µV)"); fig.colorbar(im, ax=ax[0], pad=.01)
    ax[0].set_xticks(range(18)); ax[0].set_xticklabels(CANON, rotation=90, fontsize=8); ax[0].set_yticks(range(len(VZ_S))); ax[0].set_yticklabels([s["subject"] for s in VZ_S])
    for (i_, j_), v in np.ndenumerate(M):
        if 10 ** v < 0.5: ax[0].text(j_, i_, "plat", ha="center", va="center", color="red", fontsize=7)
    ax[1].bar(np.arange(18), (10 ** M).mean(0), color=C_RAW); ax[1].set_xticks(range(18)); ax[1].set_xticklabels(CANON, rotation=90, fontsize=8)
    ax[1].set_ylabel("écart-type moyen (µV)"); ax[1].set_title("Moyenne sur les patients")
    plt.tight_layout(); vz_save("canaux_brut"); plt.show()


plot_channel_heatmaps()

#### Fenêtres de 5 s brutes, par classe

In [ ]:
def plot_window_examples(i=0, n_ch=4):
    s = VZ_S[i]; fs = s["fs"]; n = int(VZ["win_s"] * fs); t = np.arange(n) / fs; fig, ax = plt.subplots(1, 3, figsize=(18, 4), sharex=True)
    for j, (c, nm, col) in enumerate([("int", "hors crise", C_INT), ("pre", "avant crise", C_PRE), ("ict", "crise", C_ICT)]):
        k0 = max(0, (s["sig"][c].shape[1] - n) // 2); W = s["sig"][c][:n_ch, k0:k0 + n]; gap = 1.3 * np.ptp(W, 1).max() + 1e-9
        for q in range(n_ch): ax[j].plot(t, W[q] - W[q].mean() - q * gap, color=col, lw=.8)
        ax[j].set_yticks([-q * gap for q in range(n_ch)]); ax[j].set_yticklabels(CANON[:n_ch], fontsize=8); ax[j].set_xlabel("s")
        ax[j].set_title(f"{nm} (max {np.abs(s['sig'][c][:, k0:k0 + n]).max():.0f} µV)", fontsize=10, loc="left")
    fig.suptitle(f"{s['name']} — fenêtres brutes de 5 s (moyenne retirée à l'affichage, échelle propre à chaque panneau)"); plt.tight_layout(); vz_save(f"fenetres_{s['subject']}"); plt.show()


plot_window_examples(0)

#### Séparabilité brute des classes par bande
Chaque point = un patient : log10 du rapport de puissance (médiane sur les canaux). Au-dessus de 0 = plus de puissance que hors crise.

In [ ]:
def plot_separability():
    fig, ax = plt.subplots(figsize=(9, 5)); rng = np.random.default_rng(0)
    for c, col, off in (("pre", C_PRE, -.15), ("ict", C_ICT, .15)):
        for b_i, b in enumerate(VZ_BANDS):
            v = np.array([np.median(s["bp"][c][:, b_i] - s["bp"]["int"][:, b_i]) for s in VZ_S])
            ax.scatter(b_i + off + rng.uniform(-.05, .05, len(v)), v, color=col, alpha=.75, label=("avant crise / hors crise" if c == "pre" else "crise / hors crise") if b_i == 0 else None)
            ax.plot([b_i + off - .1, b_i + off + .1], [v.mean()] * 2, color="k", lw=2)
    ax.axhline(0, color="k", lw=.8); ax.set_xticks(range(5)); ax.set_xticklabels(VZ_BANDS); ax.set_title("Signal brut"); ax.legend(fontsize=9)
    ax.set_ylabel("log10 (puissance classe / puissance hors crise)"); plt.tight_layout(); vz_save("separabilite_brut"); plt.show()


plot_separability()